# 01-4. 行列分解 — 動かして確かめる

📖 解説: [`../04_decompositions.md`](../04_decompositions.md)

## このノートで触るもの
1. LU 分解と連立方程式
2. QR 分解と最小二乗法
3. Cholesky 分解
4. SVD で画像を圧縮 (低ランク近似)
5. SVD で推薦システム
6. ベンチマーク: solve vs inv の速度比較

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (04_decompositions.md)](../04_decompositions.md)

In [ ]:
import time
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from scipy.linalg import lu
from ipywidgets import interact, IntSlider

%matplotlib inline
rng = np.random.default_rng(seed=42)

## 1. LU 分解

In [ ]:
A = np.array([[2.0, 1.0, 1.0],
              [4.0, 3.0, 3.0],
              [8.0, 7.0, 9.0]])

P, L, U = lu(A)
print('L (下三角):'); print(L)
print('\nU (上三角):'); print(U)
print('\nP (置換):'); print(P)
print(f'\nP @ A ≈ L @ U: {np.allclose(P @ A, L @ U)}')

## 2. QR 分解と最小二乗法

In [ ]:
A = np.array([[1.0, 2.0],
              [3.0, 4.0],
              [5.0, 6.0]])

Q, R = np.linalg.qr(A)
print('Q:'); print(Q)
print('\nR:'); print(R)
print(f'\nQ^T Q ≈ I: {np.allclose(Q.T @ Q, np.eye(2))}')
print(f'A ≈ Q @ R: {np.allclose(A, Q @ R)}')

In [ ]:
# 最小二乗法を 3 通りで解く: 直接式 vs QR vs lstsq
n_samples = 100
X = rng.standard_normal((n_samples, 3))
true_beta = np.array([1.5, -2.0, 0.8])
y = X @ true_beta + rng.standard_normal(n_samples) * 0.1

# 方法1: 正規方程式 β = (X^T X)^(-1) X^T y (不安定)
beta_normal = np.linalg.inv(X.T @ X) @ X.T @ y

# 方法2: QR 分解
Q, R = np.linalg.qr(X)
beta_qr = np.linalg.solve(R, Q.T @ y)

# 方法3: NumPy 推奨の lstsq (内部で SVD)
beta_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)

print('真値:        ', true_beta)
print('正規方程式:  ', beta_normal)
print('QR分解:     ', beta_qr)
print('lstsq:       ', beta_lstsq)

## 3. Cholesky 分解

In [ ]:
# 対称正定値行列を作る (例: A = M^T M で必ず正定値)
M = rng.standard_normal((4, 4))
A = M.T @ M + np.eye(4)    # 対称正定値

L = np.linalg.cholesky(A)
print('L (下三角):'); print(L)
print(f'\nA ≈ L Lᵀ: {np.allclose(A, L @ L.T)}')

# 連立方程式 Ax = b を Cholesky 経由で解く
b = rng.standard_normal(4)
y = np.linalg.solve(L, b)
x = np.linalg.solve(L.T, y)
x_direct = np.linalg.solve(A, b)
print(f'\nCholesky経由の解と直接解が一致: {np.allclose(x, x_direct)}')

## 4. SVD で「画像」を圧縮 — 低ランク近似のデモ

ランダム生成の "画像" (実は構造のある行列) を SVD で圧縮します。
上位 k 個の特異値だけで、どこまで元に戻せるか体験。

In [ ]:
# 構造のある「画像」を作る (正方形に縞が入った)
size = 80
img = np.zeros((size, size))
for i in range(0, size, 10):
    img[i:i+5, :] = 1.0
img += rng.standard_normal((size, size)) * 0.1

U, s, Vt = np.linalg.svd(img, full_matrices=False)
print(f'特異値の上位10個: {s[:10]}')
print(f'特異値の総数: {len(s)}')

In [ ]:
def reconstruct(k: int) -> None:
    """上位 k 個の特異値で画像を近似復元."""
    img_approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    err = np.linalg.norm(img - img_approx) / np.linalg.norm(img)
    compress_ratio = (k * (size + size + 1)) / (size * size) * 100

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img, cmap='gray')
    axes[0].set_title('元画像')
    axes[0].axis('off')
    axes[1].imshow(img_approx, cmap='gray')
    axes[1].set_title(f'上位 {k} 個の特異値で復元 (圧縮率: {compress_ratio:.1f}%, 誤差: {err:.4f})')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

interact(
    reconstruct,
    k=IntSlider(min=1, max=80, step=1, value=5, description='特異値の数 k'),
);

In [ ]:
# 特異値スペクトル (急速に減衰しているのを見る)
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(s, 'o-')
ax.set_xlabel('インデックス')
ax.set_ylabel('特異値 (対数スケール)')
ax.set_title('特異値の急速な減衰 → 少数項で近似可能')
ax.grid(True, alpha=0.3)
plt.show()

## 5. SVD で推薦システム — 簡易 Netflix

In [ ]:
# ユーザー × 映画 の評価行列 (5段階, 0=未評価)
R = np.array([
    [5, 4, 0, 1, 0],   # ユーザー1
    [4, 0, 0, 1, 5],   # ユーザー2
    [1, 1, 0, 5, 0],   # ユーザー3
    [0, 0, 5, 4, 1],   # ユーザー4
    [5, 5, 1, 0, 0],   # ユーザー5
], dtype=float)

print('元の評価行列 (0は未評価):')
print(R)

# SVD で低ランク近似 → 欠損値の予測
U, s, Vt = np.linalg.svd(R, full_matrices=False)
k = 2  # 潜在次元 = 2
R_pred = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

print(f'\n潜在次元 k={k} で予測:')
print(np.round(R_pred, 2))

print('\n→ 0 だった所 (未評価) に予測値が入っている')
print('  例えば ユーザー1 の 映画3 は予測値', f'{R_pred[0, 2]:.2f}')

## 6. ベンチマーク: solve vs inv

「逆行列を直接計算するな、solve を使え」と言われる理由を体感します。

In [ ]:
n = 1000
A = rng.standard_normal((n, n))
b = rng.standard_normal(n)

# 方法 1: inv を使う
t0 = time.perf_counter()
x1 = np.linalg.inv(A) @ b
t_inv = time.perf_counter() - t0

# 方法 2: solve を使う
t0 = time.perf_counter()
x2 = np.linalg.solve(A, b)
t_solve = time.perf_counter() - t0

print(f'inv 経由:    {t_inv*1000:.2f} ms')
print(f'solve 直接:  {t_solve*1000:.2f} ms')
print(f'速度比:      solve は inv の {t_inv/t_solve:.2f}x 速い')

# 精度も solve のほうが高い (大きな行列ほど顕著)
err1 = np.linalg.norm(A @ x1 - b)
err2 = np.linalg.norm(A @ x2 - b)
print(f'\ninv 経由の誤差:    {err1:.2e}')
print(f'solve 経由の誤差:  {err2:.2e}')

## まとめ

ここで触ったもの:
- LU 分解
- QR 分解と最小二乗法 3手法の比較
- Cholesky 分解
- SVD による画像圧縮 (スライダー付き)
- SVD による推薦システム
- solve vs inv のベンチマーク

## 線形代数 章 完了 🎉

次の章 → [`02_calculus/README.md`](../../02_calculus/README.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [`03_eigenvalues.ipynb`](03_eigenvalues.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [次の章: 02_calculus](../../02_calculus/README.md) |